In [ ]:
! pip install numpy pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score

## 1. Recap

Let's put the notation from Weeks 4 and 5 into code.

**Feature vector.** One datapoint is a vector of $d$ real-valued features:

$$\mathbf{x} = (x_1,\, x_2,\, \ldots,\, x_d)$$

**Target value.** Each datapoint has one real-valued answer $y$.

**Training data.** A set of $n$ pairs $(\mathbf{x}_i, y_i)$ for $i = 1, \ldots, n$, where
$x_{ij}$ is the value of feature $j$ of datapoint $i$.

**The model.**

$$y = f(\mathbf{x} \mid \theta) = \theta_0 + \theta_1 x_1 + \cdots + \theta_d x_d = \theta^\top \mathbf{x}$$

**The loss.**

$$\ell(\theta) = \frac{1}{n}\sum_{i=1}^{n}\big(y_i - \hat{y}_i\big)^2$$

Everything today builds on one uncomfortable fact: **the number we minimize during
training is not the number we actually care about.**

### 1.1 Parameter Definitions

In [ ]:
RANDOM_SEED = 175   # so everyone in the room gets the same numbers

N_POINTS   = 30     # how many training points we generate
NOISE_STD  = 0.25   # how noisy those points are
MAX_DEGREE = 15     # the largest polynomial we will try
N_FOLDS    = 5      # k, for k-fold cross-validation

DEMO_DEGREE = 8     # the degree we use for the regularization demo

### 1.2 The Data

We generate data from a curve we know the answer to, so we can tell whether the model is
right. The true relationship is $y = \sin(2\pi x)$, plus random noise.

In real life you never see the true function. Here we do, which is exactly what makes
this a good teaching example.

In [ ]:
def true_function(x):
    """The relationship we are pretending not to know."""
    return np.sin(2 * np.pi * x)


def make_data(n=N_POINTS, noise=NOISE_STD, seed=RANDOM_SEED):
    """Returns n noisy samples (x_i, y_i) of the true function."""
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(0, 1, n))
    y = true_function(x) + rng.normal(0, noise, n)
    return x, y


x, y = make_data()
grid = np.linspace(0, 1, 300)   # fine grid, only used for drawing smooth curves

print(f"n = {len(x)} training pairs, d = 1 feature")

plt.figure(figsize=(7, 4))
plt.scatter(x, y, color="black", zorder=3, label="training data")
plt.plot(grid, true_function(grid), "--", color="gray", label="true function")
plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.title("What we are trying to learn")
plt.show()

### 1.3 The Loss, in Code

**Your turn.** Let's write $\ell(\theta)$ ourselves so the formula above is not abstract.

Take the difference between each true value and each prediction, square it, and average.
`np.mean` will do the averaging for you.

In [ ]:
def mse(y_true, y_pred):
    """Mean squared error: the loss we minimize during training."""
    pass

#### Let's double check our functions!

In [ ]:
assert mse(np.array([1.0, 2.0]), np.array([1.0, 2.0])) == 0.0, \
    "A perfect prediction should have zero error"
assert mse(np.array([0.0, 0.0]), np.array([1.0, 3.0])) == 5.0, \
    f"Expected 5.0 (mean of 1 and 9). Actual: {mse(np.array([0.0, 0.0]), np.array([1.0, 3.0]))}"

print("Passed")

In [ ]:
# Fit the plain linear model theta_0 + theta_1 * x
linear = LinearRegression().fit(x.reshape(-1, 1), y)

print(f"theta_0 = {linear.intercept_: .4f}")
print(f"theta_1 = {linear.coef_[0]: .4f}")
print(f"MSE     = {mse(y, linear.predict(x.reshape(-1, 1))): .4f}")

## 2. Choosing the Right Model

**Overfitting:** the model is too complex, fits every small fluctuation in the training
data, and cannot generalize to unseen data.

**Underfitting:** the model is too simple and does not match the underlying pattern.

The straight line above is underfitting badly. To see overfitting we need a model that
*can* be too flexible.

### 2.1 Making the Model Flexible: Polynomial Features

A linear model is **linear in its parameters**, not in the data. So we can introduce
additional dimensions by squaring, cubing, and so on, and still use exactly the same
fitting machinery:

$$y = f(x \mid \theta) = \theta_0 + \theta_1 x + \theta_2 x^2 + \cdots + \theta_m x^m$$

The gradient descent code from Week 5 works without a single change. We only changed
what we hand it.

**Your turn.** Build the feature matrix. A list comprehension over the powers plus
`np.column_stack` will do it in one line.

In [ ]:
def polynomial_features(x, degree):
    """
    Turn a 1D array of inputs into a matrix of polynomial features.

    x:      shape (n,)
    return: shape (n, degree), with columns x, x^2, ..., x^degree

    No column of 1s: scikit-learn's LinearRegression fits theta_0 for us.
    """
    pass

#### Let's double check our functions!

In [ ]:
test_x = np.array([2.0, 3.0])
F = polynomial_features(test_x, 3)

assert F.shape == (2, 3), f"Expected shape (2, 3). Actual: {F.shape}"
assert np.allclose(F[0], [2, 4, 8]),  f"Expected [2, 4, 8]. Actual: {F[0]}"
assert np.allclose(F[1], [3, 9, 27]), f"Expected [3, 9, 27]. Actual: {F[1]}"

print("Passed")

With that in hand, two small helpers for fitting and predicting.

In [ ]:
def fit_polynomial(x, y, degree):
    """Fit a degree-m polynomial by plain least squares."""
    model = LinearRegression()
    model.fit(polynomial_features(x, degree), y)
    return model


def predict_polynomial(model, x, degree):
    """Predict with a model that was fit on polynomial features."""
    return model.predict(polynomial_features(x, degree))

### 2.2 Seeing Both Failure Modes

Same data, three different values of $m$. Watch the training MSE in each title.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, d in zip(axes, [1, 3, 15]):
    model = fit_polynomial(x, y, d)
    train_mse = mse(y, predict_polynomial(model, x, d))

    ax.scatter(x, y, color="black", zorder=3, s=25)
    ax.plot(grid, true_function(grid), "--", color="gray")
    ax.plot(grid, predict_polynomial(model, grid, d), color="crimson", linewidth=2)
    ax.set_ylim(-2, 2)
    ax.set_title(f"m = {d}   |   train MSE = {train_mse:.4f}")
    ax.set_xlabel("x")

axes[0].set_ylabel("y")
plt.tight_layout(); plt.show()

$m = 1$ cannot bend, so it misses the shape entirely. That is **underfitting**.

$m = 15$ passes close to every point and has the lowest training MSE of the three. But
look at what it does *between* the points, especially near the edges. It is chasing
noise. That is **overfitting**.

And notice the trap: **training MSE told us $m = 15$ was the best one.**

### 2.3 Why Training Error Can Never Choose For Us

Sweep $m$ from 1 to 15 and record the training MSE at each step.

In [ ]:
degrees = np.arange(1, MAX_DEGREE + 1)
train_errors = []

for d in degrees:
    model = fit_polynomial(x, y, d)
    train_errors.append(mse(y, predict_polynomial(model, x, d)))

plt.figure(figsize=(7, 4))
plt.plot(degrees, train_errors, "o-", color="crimson", label="training MSE")
plt.xlabel("polynomial degree (m)"); plt.ylabel("MSE")
plt.title("Training error never gets worse")
plt.legend(); plt.show()

print(f"Lowest training MSE is at m = {degrees[int(np.argmin(train_errors))]}")

The curve only ever goes down, so it has no minimum in the middle. "Pick the $m$
with the lowest training error" always answers **"the biggest one."**

Adding a feature can never *hurt* the training fit, because the model can always set that
feature's weight to zero and do exactly as well as before.

**Validation data is what shows us when overfitting occurs.** We need to score the model
on data it did not train on.

## 3. Train - Validation - Test Pipeline

Split the data three ways:

| Split | What it is for | Roughly |
|---|---|---|
| **Train** | fit the parameters $\theta$ | 70% |
| **Validation** | compare models and choose between them | 15% |
| **Test** | estimate final performance, used once | 15% |

**Why do we need a test set if we already have a validation set?** Because we use the
validation set to *choose* a model. That means we are slowly, indirectly, fitting to it —
run enough comparisons and the validation score becomes optimistic too. The test set is
the only split that never influenced a decision.

### 3.1 Splitting the Data

**Your turn.** Write the split. Some hints:

* `rng.permutation(n)` gives a shuffled array of the indices `0 .. n-1`
* the first `n_val` of those become validation, the rest become training
* return four arrays: `x_train, y_train, x_val, y_val`

In [ ]:
def train_val_split(x, y, val_fraction=0.3, seed=RANDOM_SEED):
    """
    Randomly split the data into a training part and a validation part.

    Returns: x_train, y_train, x_val, y_val
    """
    pass

#### Let's double check our functions!

In [ ]:
x_tr, y_tr, x_va, y_va = train_val_split(x, y, val_fraction=0.3)

assert len(x_tr) + len(x_va) == len(x), "Every point should land in exactly one side"
assert len(x_va) == 9,  f"Expected 9 validation points out of 30. Actual: {len(x_va)}"
assert len(x_tr) == 21, f"Expected 21 training points out of 30. Actual: {len(x_tr)}"
assert len(np.intersect1d(x_tr, x_va)) == 0, "No point may appear on both sides"

print("Passed")

### 3.2 Validation Error Reveals the Overfitting

Sweep $m$ again, but this time record **both** errors.

In [ ]:
train_errors, val_errors = [], []

for d in degrees:
    model = fit_polynomial(x_tr, y_tr, d)
    train_errors.append(mse(y_tr, predict_polynomial(model, x_tr, d)))
    val_errors.append(mse(y_va, predict_polynomial(model, x_va, d)))

plt.figure(figsize=(7, 4.5))
plt.plot(degrees, train_errors, "o-", color="crimson",   label="training MSE")
plt.plot(degrees, val_errors,   "s-", color="steelblue", label="validation MSE")
plt.yscale("log")   # log scale, because the right-hand side explodes
plt.xlabel("polynomial degree (m)"); plt.ylabel("MSE (log scale)")
plt.title("Underfitting on the left, overfitting on the right")
plt.legend(); plt.show()

print(f"Best m by validation error: {degrees[int(np.argmin(val_errors))]}")

Training error slides downhill forever. Validation error comes down, bottoms out,
then climbs steeply. The bottom of that U is the model we want.

A diagnostic table worth keeping:

| | High training error | Low training error |
|---|---|---|
| **High validation error** | Underfitting | Overfitting |
| **Low validation error** | Something is wrong — check your code | Good fit |

But there is a problem with what we just did. Our validation set has **9 points in it**.
Nine. Whether $m = 3$ beats $m = 5$ may come down to which particular nine points got
shuffled into that set.

## 4. K-Fold Cross Validation

Instead of one split, make $k$ of them. Divide the data into $k$ equal **folds**, then:

* train on folds 2, 3, 4, 5 → score on fold 1
* train on folds 1, 3, 4, 5 → score on fold 2
* ... $k$ times in total

Every point is used for validation exactly once, and we average the $k$ scores. The cost
is that we train the model $k$ times instead of once.

Setting $k = n$ is called **leave-one-out** cross-validation: maximum training data per
fit, but $n$ separate fits.

### 4.1 Building the Folds

**Your turn.** Write the function that produces the $k$ (train, validation) index pairs. Hints:

* shuffle the indices `0 .. n-1` first
* `np.array_split(perm, k)` chops a shuffled array into `k` roughly equal pieces
* for fold `i`: that piece is validation, and everything else concatenated
  (`np.concatenate`) is training

In [ ]:
def k_fold_indices(n_samples, k, seed=RANDOM_SEED):
    """
    Build the index pairs for k-fold cross-validation.

    Returns a list of k tuples: [(train_idx, val_idx), ...]
    """
    pass

#### Let's double check our functions!

In [ ]:
folds = k_fold_indices(20, 5)

assert len(folds) == 5, f"Expected 5 folds. Actual: {len(folds)}"

# Every point must be validated exactly once across all folds
all_val = np.concatenate([v for _, v in folds])
assert len(all_val) == 20, f"Expected 20 validation points total. Actual: {len(all_val)}"
assert len(np.unique(all_val)) == 20, "Each point must be validated exactly once"

# Within a fold, train and validation must not overlap
for train_idx, val_idx in folds:
    assert len(train_idx) + len(val_idx) == 20, "Every point belongs to one side"
    assert len(np.intersect1d(train_idx, val_idx)) == 0, "No leakage between sides"

print("Passed")

### 4.2 What the Folds Look Like

Let's print the actual partition, so the illustration from the notes becomes concrete.

In [ ]:
for i, (train_idx, val_idx) in enumerate(k_fold_indices(20, N_FOLDS), start=1):
    marks = ["train"] * 20
    for j in val_idx:
        marks[j] = "  VAL"
    print(f"fold {i}: " + " ".join(m[-3:] for m in marks))

print("\nEach column is one datapoint. Read down a column: it is VAL exactly once.")

## 5. Cross Validation for Regression

Now we use the folds to answer the question we have been stuck on since Section 2:
**which $m$ do we pick?**

### 5.1 The Cross-Validation Loop

**Your turn.** This is the centrepiece of the session. Use `k_fold_indices` from Section 4
to get the fold pairs, then for each pair:

1. slice out `x_tr, y_tr` and `x_va, y_va` using the two index arrays
2. fit on the training part with `fit_polynomial`
3. score on the validation part with `mse`
4. collect that score

Return the mean, the standard deviation, and **every individual fold score** — not just
the mean. You will see why in a moment.

In [ ]:
def cross_val_mse(x, y, degree, k=N_FOLDS, alpha=0.0, kind="ridge"):
    """
    Estimate a model's generalization error with k-fold cross-validation.

    alpha=0.0 means no regularization (we use this until Section 6).
    When alpha is not 0, fit with fit_regularized / predict_regularized instead.

    Returns (mean_mse, std_mse, all_fold_scores)
    """
    pass

#### Let's double check our functions!

In [ ]:
mean_mse, std_mse, fold_scores = cross_val_mse(x, y, degree=3)

assert len(fold_scores) == N_FOLDS, \
    f"Expected {N_FOLDS} fold scores. Actual: {len(fold_scores)}"
assert np.isclose(mean_mse, np.mean(fold_scores)), \
    "The first return value should be the mean of the fold scores"
assert np.all(np.array(fold_scores) >= 0), "Squared error is never negative"

print("Passed")

In [ ]:
for d in [1, 3, 5, 9, 15]:
    mean_mse, std_mse, fold_scores = cross_val_mse(x, y, d)
    pretty = "  ".join(f"{s:8.2f}" for s in fold_scores)
    print(f"m = {d:2d} | CV MSE = {mean_mse:9.3f} +/- {std_mse:8.3f} | folds: {pretty}")

Look at the **$m = 15$** row, and look at the individual folds rather than the mean.

On one fold it scores about 0.36 — better than $m = 9$ manages. On another it scores in
the hundreds. The model is not consistently bad; it is *wildly inconsistent*.

That spread across folds is the fingerprint of overfitting, and it is why we report the
standard deviation alongside the mean. A model whose fold scores disagree by two orders
of magnitude has not learned the signal. It memorized whichever points it happened to
see.

(This is also the seed of the **bias-variance tradeoff**, coming later in the curriculum.)

### 5.2 Choosing m

In [ ]:
cv_means, cv_stds = [], []

for d in degrees:
    mean_mse, std_mse, _ = cross_val_mse(x, y, d)
    cv_means.append(mean_mse); cv_stds.append(std_mse)

cv_means = np.array(cv_means); cv_stds = np.array(cv_stds)

plt.figure(figsize=(7, 4.5))
plt.errorbar(degrees, cv_means, yerr=cv_stds, fmt="o-", color="seagreen",
             capsize=4, label="CV MSE (+/- 1 std across folds)")
plt.yscale("log")
plt.xlabel("polynomial degree (m)"); plt.ylabel("CV MSE (log scale)")
plt.title(f"{N_FOLDS}-fold cross-validation")
plt.legend(); plt.show()

best_degree = int(degrees[np.argmin(cv_means)])
print(f"Best m by cross-validation: {best_degree}")

### 5.3 Checking Against scikit-learn

We wrote the loop by hand so we know exactly what it does. In practice you would call
`cross_val_score`, which does the same thing.

Note the sign: scikit-learn *maximizes* its scores, so it reports **negative** MSE.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

for d in [1, 3, 9]:
    ours = cross_val_mse(x, y, d)[0]
    sk = -cross_val_score(LinearRegression(), polynomial_features(x, d), y,
                          scoring="neg_mean_squared_error", cv=kf)
    print(f"m = {d:2d} | ours: {ours:8.4f} | sklearn: {sk.mean():8.4f}")

print("\nThey differ slightly because the folds are shuffled differently.")
print("The conclusion they point to is the same, which is what matters.")

## 6. Regularization

So far "complexity" has meant $m$ — a dial we turn from outside the model. But here is a
better question:

> What if there was a way to quantify how complex a model is using the parameter $\theta$
> itself?

### 6.1 Look at Theta

Fit a degree-8 polynomial with no regularization and just print the weights.

In [ ]:
model_8 = fit_polynomial(x, y, DEMO_DEGREE)

print("Unregularized degree-8 coefficients:\n")
for p, c in enumerate(model_8.coef_, start=1):
    print(f"  theta_{p} (x^{p}):  {c:12.1f}")

Those numbers are enormous, and they **alternate in sign**. A coefficient near
$+2000$ sits right next to one near $-1860$. They are fighting each other, and their
near-cancellation is what lets the curve whip up and down through every noisy point.

That is what overfitting looks like from the inside: **huge weights**.

So define a function $R(\theta)$ that is large when the weights are large, and add it to
the loss:

$$\ell_R(\theta) = \underbrace{\frac{1}{n}\sum_i (y_i - \hat{y}_i)^2}_{\text{fit the data}}
\;+\; \alpha \underbrace{R(\theta)}_{\text{stay simple}}$$

What should $R$ be? That is what norms are for.

## 7. Norms

A **norm** is a function that takes in a vector and measures its size or magnitude. The
norm of a vector $\mathbf{v}$ is written $\lVert \mathbf{v} \rVert$, and it is always a
non-negative real number.

Two of them matter here:

$$\lVert \theta \rVert_1 = \sum_j |\theta_j| \qquad\qquad
  \lVert \theta \rVert_2 = \sqrt{\sum_j \theta_j^2}$$

| Penalty | Norm used | Name |
|---|---|---|
| **L2** | $\lVert\theta\rVert_2^2$ | Ridge |
| **L1** | $\lVert\theta\rVert_1$ | LASSO |

### 7.1 Writing the Norms

**Your turn.** Two short functions. Note that Ridge penalizes the **squared** L2 norm, not the norm
itself — we will see in Section 7.4 exactly why that squaring matters.

In [ ]:
def l1_norm(v):
    """Sum of absolute values."""
    pass


def l2_norm(v):
    """Square root of the sum of squares."""
    pass

#### Let's double check our functions!

In [ ]:
v = np.array([3.0, -4.0])

assert l1_norm(v) == 7.0, f"Expected 7.0 (3 + 4). Actual: {l1_norm(v)}"
assert l2_norm(v) == 5.0, f"Expected 5.0 (the 3-4-5 triangle). Actual: {l2_norm(v)}"
assert l1_norm(np.zeros(5)) == 0.0, "The zero vector has zero size"

print("Passed")

print(f"\nOur overfit degree-8 model has ||theta||_1 = {l1_norm(model_8.coef_):.1f}")
print(f"                              and ||theta||_2 = {l2_norm(model_8.coef_):.1f}")
print("\nThat is the 'complexity' number we now want to push down.")

### 7.2 The Constrained View

There is a second way to write the same idea. Instead of adding a penalty, put a hard
budget on the size of $\theta$:

$$\min_\theta \; \ell(\theta) \quad \text{subject to} \quad \lVert\theta\rVert_1 \le t$$

These two forms are the same problem. Build the Lagrangian for the constrained version:

$$\mathcal{L}(\theta, \lambda) = \ell(\theta) + \lambda\big(\lVert\theta\rVert_1 - t\big)$$

Setting the gradient to zero gives $\nabla \ell + \lambda \nabla R = 0$, which is exactly
the stationarity condition of the penalized form with $\alpha = \lambda$. **So $\alpha$
is a Lagrange multiplier.** Turning up the penalty and tightening the budget are the same
act described two ways.

### 7.3 Fitting With Each Penalty

One practical note first: the penalty is **not scale invariant**. The column $x^8$ lives
on a completely different scale from the column $x$, so an unscaled penalty would punish
them unequally for no good reason. We standardize the features first.

In [ ]:
def fit_regularized(x, y, degree, alpha, kind="ridge"):
    """
    Fit a regularized polynomial.

    kind: "ridge" for the L2 penalty, "lasso" for L1.
    Returns (model, scaler) — we need the scaler again at predict time.
    """
    F = polynomial_features(x, degree)

    scaler = StandardScaler().fit(F)   # put every feature column on the same scale
    F_scaled = scaler.transform(F)

    if kind == "ridge":
        model = Ridge(alpha=alpha)
    else:
        model = Lasso(alpha=alpha, max_iter=200000)

    model.fit(F_scaled, y)
    return model, scaler


def predict_regularized(model, scaler, x, degree):
    """Predict with a regularized model (remember to scale first)."""
    return model.predict(scaler.transform(polynomial_features(x, degree)))

In [ ]:
ridge_model, ridge_scaler = fit_regularized(x, y, DEMO_DEGREE, alpha=1.0,  kind="ridge")
lasso_model, lasso_scaler = fit_regularized(x, y, DEMO_DEGREE, alpha=0.01, kind="lasso")

comparison = pd.DataFrame({
    "feature":    [f"x^{p}" for p in range(1, DEMO_DEGREE + 1)],
    "no penalty": np.round(model_8.coef_, 1),
    "L2 (Ridge)": np.round(ridge_model.coef_, 3),
    "L1 (LASSO)": np.round(lasso_model.coef_, 3),
})

print(comparison.to_string(index=False))

Compare the two penalized columns carefully.

**Ridge** shrank everything. Every coefficient is small now, but every coefficient is
still *there*.

**LASSO** did something different. Several coefficients are **exactly zero**. Not small,
not `1e-9` — zero. LASSO didn't just shrink the model, it *deleted features from it*.
That property is called **sparsity**.

### 7.4 Why L1 Gives Exact Zeros

The reason is in the derivatives.

$$\frac{d}{d\theta}\,\theta^2 = 2\theta
\qquad\qquad
\frac{d}{d\theta}\,|\theta| = \begin{cases} +1 & \theta > 0 \\ -1 & \theta < 0\end{cases}$$

**L2's pull toward zero is proportional to the weight itself.** As $\theta_j$ shrinks the
force shrinks with it, so the weight approaches zero and never arrives.

**L1's pull is a constant $\alpha$**, no matter how small the weight already is. A weight
that is not earning its keep gets pushed all the way to zero and held there.

At $\theta = 0$ itself, $|\theta|$ is **not differentiable**. Check the difference
quotient from each side:

In [ ]:
def difference_quotient(f, theta, h):
    """The slope of the line through (theta, f(theta)) and (theta+h, f(theta+h))."""
    return (f(theta + h) - f(theta)) / h


print("Approaching 0 in |theta|:")
for h in [0.1, 0.01, 0.001]:
    right = difference_quotient(abs, 0.0,  h)
    left  = difference_quotient(abs, 0.0, -h)
    print(f"  h = {h:6.3f} | from the right: {right:+.1f} | from the left: {left:+.1f}")

print("\nThe two sides disagree, so the derivative at 0 does not exist. That kink")
print("is a stable resting place: any small move away is penalized immediately.\n")

print("Now the same test on theta^2:")
for h in [0.1, 0.01, 0.001]:
    right = difference_quotient(lambda t: t ** 2, 0.0,  h)
    left  = difference_quotient(lambda t: t ** 2, 0.0, -h)
    print(f"  h = {h:6.3f} | from the right: {right:+.3f} | from the left: {left:+.3f}")

print("\nBoth sides agree and go to 0. Squaring removed the kink, which is exactly")
print("why L2 gives a smooth solution and never an exact zero.")

### 7.5 Counting What Survived

**Your turn.** Count how many coefficients are meaningfully different from zero.

Why the tolerance? Floating point arithmetic rarely produces a clean `0.0`. Anything
below `tol` in absolute value should count as zero. `np.abs` and `np.sum` are all you
need.

In [ ]:
def count_nonzero_coefs(coefs, tol=1e-8):
    """Count coefficients meaningfully different from zero (floating point is messy)."""
    pass

#### Let's double check our functions!

In [ ]:
assert count_nonzero_coefs(np.array([0.0, 1.0, -2.0, 0.0])) == 2, "Expected 2"
assert count_nonzero_coefs(np.array([1e-12, 5.0])) == 1, "Expected 1 (1e-12 counts as zero)"
assert count_nonzero_coefs(np.zeros(8)) == 0, "Expected 0"

print("Passed")

print(f"\nRidge kept {count_nonzero_coefs(ridge_model.coef_)} of {DEMO_DEGREE} features")
print(f"LASSO kept {count_nonzero_coefs(lasso_model.coef_)} of {DEMO_DEGREE} features")

### 7.6 Turning the Alpha Dial

Watch what $\alpha$ does to the fitted curve.

In [ ]:
alphas = [0.0, 1e-4, 1e-2, 1.0, 100.0]
fig, axes = plt.subplots(1, len(alphas), figsize=(19, 3.6), sharey=True)

for ax, a in zip(axes, alphas):
    if a == 0.0:
        m = fit_polynomial(x, y, DEMO_DEGREE)
        curve = predict_polynomial(m, grid, DEMO_DEGREE)
    else:
        m, s = fit_regularized(x, y, DEMO_DEGREE, alpha=a, kind="ridge")
        curve = predict_regularized(m, s, grid, DEMO_DEGREE)

    ax.scatter(x, y, color="black", s=18, zorder=3)
    ax.plot(grid, true_function(grid), "--", color="gray")
    ax.plot(grid, curve, color="darkorange", linewidth=2)
    ax.set_ylim(-2, 2); ax.set_title(f"alpha = {a}"); ax.set_xlabel("x")

axes[0].set_ylabel("y")
plt.tight_layout(); plt.show()

Left to right: wiggly, reasonable, reasonable, smoother, flat. $\alpha$ moves along
the *same* underfit-to-overfit axis that $m$ did — but continuously, without changing the
model's functional form.

### 7.7 Choosing Alpha the Right Way

We picked `alpha=1.0` and `alpha=0.01` above because they looked good. That should bother
you — it is exactly the mistake Section 2 warned about.

But we already built the tool that fixes it. Cross-validation can choose $\alpha$, and it
can choose $m$ at the same time.

In [ ]:
degree_grid = [1, 3, 5, 7, 9, 11]
alpha_grid  = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]

results = np.zeros((len(degree_grid), len(alpha_grid)))
for i, d in enumerate(degree_grid):
    for j, a in enumerate(alpha_grid):
        results[i, j] = cross_val_mse(x, y, d, alpha=a, kind="ridge")[0]

table = pd.DataFrame(np.round(results, 4),
                     index=[f"m = {d}" for d in degree_grid],
                     columns=[f"alpha={a:g}" for a in alpha_grid])
print(table.to_string())

bi, bj = np.unravel_index(np.argmin(results), results.shape)
print(f"\nBest combination: m = {degree_grid[bi]}, alpha = {alpha_grid[bj]:g} "
      f"(CV MSE {results[bi, bj]:.4f})")

Read across the top row: at $m = 1$, $\alpha$ barely matters — the model was never
flexible enough to overfit, so there is nothing to rein in. Read down the high-$m$ rows:
regularization is doing real work there.

The two dials interact, which is why you search over pairs rather than tuning them one at
a time.

**This is the whole session in one table.** There are two levels of optimization:

| Level | What we choose | How | Which data |
|---|---|---|---|
| Inner | parameters $\theta$ | minimize MSE (gradient descent) | training folds |
| Outer | hyperparameters $m$, $\alpha$ | minimize CV error | held-out folds |

Weeks 4 and 5 taught you the inner level. Today is the outer level.

## 8. Real Data: Penguins

Our sine wave was synthetic and we knew the answer. Let's finish on real measurements,
where we don't.

We predict a penguin's **body mass** from its **flipper length**, and let
cross-validation tell us what $m$ to use.

In [ ]:
penguins = pd.read_csv("penguins.csv").dropna(subset=["flipper_length_mm", "body_mass_g"])

px = penguins["flipper_length_mm"].to_numpy(dtype=float)
py = penguins["body_mass_g"].to_numpy(dtype=float)

# Standardize x and put y in kilograms, so high powers of x stay numerically sane.
px = (px - px.mean()) / px.std()
py = py / 1000.0

print(f"{len(px)} penguins")

plt.figure(figsize=(7, 4))
plt.scatter(px, py, alpha=0.5, color="black", s=18)
plt.xlabel("flipper length (standardized)"); plt.ylabel("body mass (kg)")
plt.title("Real data, real noise")
plt.show()

In [ ]:
print("  m  |   CV MSE   |  std across folds")
print("-" * 42)

for d in range(1, 9):
    mean_mse, std_mse, _ = cross_val_mse(px, py, d)
    print(f" {d:2d}  |  {mean_mse:8.4f}  |  {std_mse:8.4f}")

Read that table carefully, because it teaches something better than a clean answer
would.

The lowest mean sits somewhere in the middle. But look at the standard deviation column:
the spread across folds is roughly **twice as large as the gap between the best and worst
values of $m$**. Those models are not meaningfully different — the ranking is mostly
noise.

When several models sit within one standard error of the best, the usual rule is to
**take the simplest one.** Here that means a straight line.

So the honest conclusion: flipper length and body mass really are close to linearly
related, and the extra polynomial terms buy nothing. Cross-validation is not a machine
for justifying complicated models. Quite often it tells you the simple one was right, and
that is a useful thing for it to be able to tell you.

## 9. Recap

1. **Training error always decreases** with complexity, so it can never choose a model.
2. **Validation data** reveals overfitting. The test set exists because choosing on
   validation slowly fits to it.
3. **K-fold cross-validation** rotates which fold is held out, so every point is used for
   validation exactly once. It is how we pick $m$.
4. **Regularization** measures complexity from inside the model, using a **norm** of
   $\theta$. L2 (Ridge) shrinks every weight; L1 (LASSO) sets some to exactly zero,
   because $|\theta|$ has a kink at 0 and $\theta^2$ does not.
5. $\alpha$ is a **Lagrange multiplier** — penalizing and budgeting are the same problem.
6. Always look at the **spread across folds**, not just the mean.

### Try on your own

* Set `NOISE_STD = 0.05` and re-run. Does the best $m$ go up or down? Why?
* Set `N_POINTS = 200`. Does the overfitting get better or worse?
* Swap Ridge for LASSO in the Section 7.7 grid. Does it choose a different pair?
* Set `N_FOLDS = len(x)`. That is leave-one-out cross-validation. Time it.